# ЛР3. Эксперимент 3 на Kaggle

Сравниваем ответы Qwen для Dense и RRF с переранжированием MiniLM на holdout. Обе серии используют те же 210 утверждений, по 3 документа и бюджет входа 2048 токенов. Запускать ячейки сверху вниз на T4 с включённым Internet. Заранее подключить приватный Dataset с lab3_holdout_inputs.zip из запуска 8abf0dc9.

## 1. Условия запуска

Вставь полный хеш коммита, который добавляет клиент ЛР3. Если Kaggle выделил две T4, процесс увидит одну.

In [ ]:
import os
import sys
import json
import shutil
import hashlib
import subprocess
import zipfile
from pathlib import Path

ART = Path("/kaggle/working/lab3_holdout_artifacts")
INPUTS = ART / "inputs"
REPO = Path("/tmp/intsys-labs-lab3")
PYTHON = "/tmp/lab3-env/bin/python"
ART.mkdir(parents=True, exist_ok=True)

ENV = dict(os.environ)
ENV.pop("PYTHONPATH", None)
ENV.update(CUDA_VISIBLE_DEVICES="0", HF_HOME="/tmp/lab3-hf",
           PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1",
           OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg",
           LAB3_ARTIFACTS=str(ART))

def cmd(arguments, cwd=None, show=True):
    process = subprocess.Popen(arguments, cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            lines.append(line)
            if show:
                print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        if process.poll() is None:
            process.terminate()
            process.wait(timeout=30)
        raise
    text = "".join(lines)
    if code:
        if not show:
            print(text)
        raise RuntimeError(f"Команда завершилась с кодом {code}: {arguments}")
    return text

COMMIT = input("Вставь полный хеш из git rev-parse HEAD: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT), "Нужны 40 шестнадцатеричных символов"
print("Коммит запуска:", COMMIT)
print("Python блокнота:", sys.version)
cmd(["nvidia-smi"])
gpu = cmd(["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"], show=False).strip()
assert "T4" in gpu, f"Ожидали T4, получено: {gpu}"
print("Используем только GPU 0:", gpu)

## 2. Проверка входов

Сверяем контрольные суммы и копируем входные файлы в рабочую папку.

In [ ]:
ZIP_SHA = "04d4c2b65b22684655ac5fb324ebffbbed7052c68ad0c65b8612e35ddfabbe49"
META_SHA = "4832505efc0d92e5e4128bc1794ad040c9a20475b9fadacd53e3e09c2bebcc13"
FILES = ("metadata.json", "tasks.jsonl", "warmups.jsonl", "generation_config.json", "answer_schema.json", "selection.json")

def digest(data):
    return hashlib.sha256(data).hexdigest()

input_root = Path("/kaggle/input")
zip_paths = [path for path in input_root.rglob("lab3_holdout_inputs.zip")
             if digest(path.read_bytes()) == ZIP_SHA]
if zip_paths:
    with zipfile.ZipFile(zip_paths[0]) as archive:
        files = {name: archive.read(name) for name in FILES}
else:
    matches = [path for path in input_root.rglob("metadata.json")
               if digest(path.read_bytes()) == META_SHA]
    assert matches, "Подключи Dataset с архивом lab3_holdout_inputs.zip из запуска 8abf0dc9"
    files = {name: (matches[0].parent / name).read_bytes() for name in FILES}

assert digest(files["metadata.json"]) == META_SHA
preparation = json.loads(files["metadata.json"])
assert preparation["tasks"] == 420 and preparation["split"] == "holdout"
INPUTS.mkdir(parents=True, exist_ok=True)
for name, data in files.items():
    if name != "metadata.json":
        assert digest(data) == preparation["files_sha256"][name], name
    target = INPUTS / name
    if target.exists():
        assert target.read_bytes() == data, f"Изменился существующий файл {name}"
    else:
        target.write_bytes(data)
print("Входы проверены. Заданий:", preparation["tasks"])

## 3. Загрузка кода

Берём зафиксированный коммит из GitHub.

In [ ]:
if not REPO.exists():
    cmd(["git", "clone", "https://github.com/manaer6-alt/intsys-labs.git", str(REPO)])
cmd(["git", "-C", str(REPO), "fetch", "origin", COMMIT])
cmd(["git", "-C", str(REPO), "checkout", "--detach", COMMIT])
assert cmd(["git", "-C", str(REPO), "rev-parse", "HEAD"], show=False).strip() == COMMIT
assert not cmd(["git", "-C", str(REPO), "status", "--porcelain"], show=False).strip()

assert (REPO / "lab3/inference_holdout.py").is_file(), "В коммите отсутствует клиент ЛР3"

## 4. Подготовка окружения

Используем зависимости ЛР2. Проверяем, что PyTorch видит одну T4 и может выполнить CUDA-операцию.

In [ ]:
UV = shutil.which("uv")
if UV is None:
    cmd([sys.executable, "-m", "pip", "install", "uv"])
    UV = shutil.which("uv")
UV_CMD = [UV] if UV else [sys.executable, "-m", "uv"]
if not Path(PYTHON).exists():
    cmd(UV_CMD + ["venv", "--python", "3.13", "--seed", "/tmp/lab3-env"])
cmd(UV_CMD + ["pip", "install", "--python", PYTHON,
              "-r", str(REPO / "lab2/requirements-gpu.txt"), "--torch-backend=auto", "torch==2.14.0+cu132"])
cmd(UV_CMD + ["pip", "check", "--python", PYTHON])
freeze = cmd(UV_CMD + ["pip", "freeze", "--python", PYTHON], show=False)
(ART / "requirements_gpu.txt").write_text(freeze, encoding="utf-8")
(ART / "uv_version.txt").write_text(cmd(UV_CMD + ["--version"], show=False), encoding="utf-8")

check_code = r'''
import json, os, platform
from pathlib import Path
from importlib.metadata import version
import torch

assert torch.cuda.is_available(), "CUDA недоступна"
assert torch.cuda.device_count() == 1, "Процесс должен видеть только одну GPU"
assert "T4" in torch.cuda.get_device_name(0), "Нужна T4"
assert torch.__version__ == "2.14.0+cu132", f"PyTorch отличается от ЛР2: {torch.__version__}"
torch.set_num_threads(1)
value = torch.ones(1, device="cuda").cpu().item()
record = {
    "python": platform.python_version(),
    "versions": {name: version(name) for name in (
        "vllm", "torch", "transformers", "huggingface-hub", "tokenizers", "numpy", "openai", "PyYAML", "wrapt"
    )},
    "torch_cuda": torch.version.cuda, "cuda_device_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_memory_bytes": torch.cuda.get_device_properties(0).total_memory,
    "cuda_test_value": value, "cpu_logical_count": os.cpu_count(),
    "environment_variables": {k: os.environ.get(k) for k in (
        "CUDA_VISIBLE_DEVICES", "OMP_NUM_THREADS", "TOKENIZERS_PARALLELISM"
    )},
}
(Path(os.environ["LAB3_ARTIFACTS"]) / "environment.json").write_text(
    json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(record, ensure_ascii=False, indent=2))
'''
cmd([PYTHON, "-c", check_code])

## 5. Запуск Qwen через vLLM

Переиспользуем код запуска сервера ЛР2, чтобы сохранить FP16, одну T4, ревизию модели и выключенный кеш префикса.

In [ ]:
sys.path.insert(0, str(REPO / "lab2"))
from kaggle_control import KaggleRunner

class HoldoutRunner(KaggleRunner):
    def __init__(self):
        self.inputs, self.root, self.python, self.env = INPUTS, ART, PYTHON, ENV
        self.config = json.loads((INPUTS / "generation_config.json").read_text(encoding="utf-8"))
        assert self.config == json.loads((REPO / "configs/lab2.json").read_text(encoding="utf-8"))
        self.process = None

    def checkpoint(self, label):
        import uuid

        summary = {}
        selection = ART / "selection.json"
        if selection.exists():
            for stage, run_id in json.loads(selection.read_text(encoding="utf-8")).items():
                run_dir = ART / "runs" / run_id
                meta = json.loads((run_dir / "metadata.json").read_text(encoding="utf-8"))
                metrics_path = run_dir / "metrics.json"
                metrics = json.loads(metrics_path.read_text(encoding="utf-8")) if metrics_path.exists() else {}
                summary[stage] = {"run_id": run_id, "status": meta["status"],
                                  "recorded": meta.get("recorded", 0), **metrics}
        (ART / "execution_summary.json").write_text(
            json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        target = ART.parent / f"lab3_holdout_{label}_{uuid.uuid4().hex[:8]}.zip"
        with zipfile.ZipFile(target, "x", zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(ART.rglob("*")):
                if path.is_file() and path.name != "inference.lock":
                    archive.write(path, path.relative_to(ART).as_posix())
            for name in ("lab3/inference_holdout.py", "lab3/inference_dev.py", "lab3/selection.json",
                         "lab2/run_inference.py", "lab2/kaggle_control.py",
                         "lab2/answers.py", "lab2/requirements-gpu.txt", "configs/lab2.json"):
                archive.write(REPO / name, "provenance/" + name)
        with zipfile.ZipFile(target) as archive:
            assert archive.testzip() is None
        target.with_suffix(".zip.sha256").write_text(
            digest(target.read_bytes()) + "  " + target.name + "\n", encoding="ascii"
        )
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        print("Архив:", target)
        return target

    def stage(self, name):
        try:
            cmd([PYTHON, "-B", "-u", str(REPO / "lab3/inference_holdout.py"),
                 "--inputs", str(INPUTS), "--root", str(ART), "--stage", name], cwd=REPO)
        except (RuntimeError, KeyboardInterrupt):
            self.checkpoint(f"interrupted_{name}")
            raise

runner = HoldoutRunner()
runner.start_server()

## 6. Проверка и прогрев

Сверяем токенизацию 420 основных и двух проверочных входов. Qwen генерирует два ответа на утверждения из claims_train.

In [ ]:
runner.stage("check")

## 7. Серия Dense

210 запросов с документами из выдачи Dense. Сохраняем промежуточный архив.

In [ ]:
runner.stage("dense")
dense_archive = runner.checkpoint("dense")

## 8. Серия RRF + MiniLM

210 запросов с документами из выдачи RRF после переранжирования MiniLM. Скачай финальный ZIP и его SHA-256 из Output, затем заверши GPU-сессию.

In [ ]:
runner.stage("rrf_ce")
final_archive = runner.checkpoint("final")
runner.stop_server()